# 🎬 쇼츠 재료 만들기 (구글 코랩)

구글 드라이브에 있는 **영화 원본(고화질·긴 파일도 OK)**에서 쓸 구간만 잘라서,
PC방 크롬에서 가볍게 열리는 **720p 재료 영상 + 맞춘 자막**을 드라이브 `쇼츠재료` 폴더에 저장해요.

### 사용법
1. 아래 **① 칸 왼쪽의 ▶ 버튼**을 누르세요 → 구글 계정 연결을 허용하면 드라이브 안의 영화·자막 목록이 번호와 함께 나와요.
2. **② 칸**에 영화 번호, 자막 번호, 시작·끝 시간을 넣고 ▶ 버튼을 누르세요.
3. 끝나면 쇼츠 제작기에서 `쇼츠재료` 폴더의 mp4와 srt를 불러오면 돼요. (자막 시작 시간은 0:00:00)

※ 영화는 다운로드할 필요 없이 드라이브에 올려 두기만 하면 돼요.

In [ ]:
#@title ① 드라이브 연결하고 영화 찾기 { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, subprocess, html, time

VIDEO_EXT = ('.mp4', '.mkv', '.avi', '.mov', '.m4v', '.ts', '.wmv', '.webm')
SUB_EXT = ('.smi', '.srt', '.ass', '.ssa', '.vtt')
OUT_DIR_NAME = '쇼츠재료'


def find_files(root, max_files=300):
    """드라이브 안의 영상·자막 파일을 찾아요 (휴지통·숨김 폴더 제외)."""
    vids, subs = [], []
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if not x.startswith('.') and x not in ('Colab Notebooks', OUT_DIR_NAME)]
        for f in files:
            p = os.path.join(d, f); low = f.lower()
            if low.endswith(VIDEO_EXT): vids.append(p)
            elif low.endswith(SUB_EXT): subs.append(p)
        if len(vids) + len(subs) > max_files: break
    vids.sort(key=lambda p: os.path.basename(p).lower()); subs.sort(key=lambda p: os.path.basename(p).lower())
    return vids, subs


def hms(t):
    t = max(0, int(round(t))); return f'{t//3600}:{t%3600//60:02d}:{t%60:02d}'


def parse_time(s):
    s = str(s).strip().replace('：', ':')
    if not s: return None
    parts = [float(x) for x in s.split(':')]
    v = 0
    for x in parts: v = v * 60 + x
    return v


def read_text(path):
    b = open(path, 'rb').read()
    for enc in ('utf-8-sig', 'cp949', 'euc-kr', 'utf-16'):
        try:
            t = b.decode(enc)
            if enc.startswith('utf-8') and '�' in t: continue
            return t
        except Exception:
            pass
    return b.decode('utf-8', 'ignore')


def _clean(t):
    t = re.sub(r'(?i)<br\s*/?>', '\n', t)
    t = re.sub(r'<[^>]+>', '', t)
    t = html.unescape(t).replace('\xa0', ' ')
    t = '\n'.join(x.strip() for x in t.split('\n'))
    return re.sub(r'\n+', '\n', t).strip()


def parse_smi(txt):
    out = []
    items = re.findall(r'(?is)<sync\s+start\s*=\s*["\']?(\d+)[^>]*>(.*?)(?=<sync|</body|$)', txt)
    # 한국어 자막만 (여러 언어가 섞인 smi)
    has_kr = any(re.search(r'(?i)class\s*=\s*["\']?(krcc|kr)', b) for _, b in items)
    for i, (ms, body) in enumerate(items):
        if has_kr:
            m = re.search(r'(?is)<p[^>]*class\s*=\s*["\']?(krcc|kr)[^>]*>(.*)', body)
            body = m.group(2) if m else ''
        text = _clean(body)
        st = int(ms) / 1000
        en = int(items[i + 1][0]) / 1000 if i + 1 < len(items) else st + 3
        if text: out.append({'start': st, 'end': min(en, st + 10), 'text': text})
    return out


def parse_srt(txt):
    out = []
    for blk in re.split(r'\n\s*\n', txt.replace('\r', '')):
        m = re.search(r'(\d+):(\d+):(\d+)[,.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,.](\d+)', blk)
        if not m: continue
        g = [int(x) for x in m.groups()]
        st = g[0] * 3600 + g[1] * 60 + g[2] + g[3] / 1000
        en = g[4] * 3600 + g[5] * 60 + g[6] + g[7] / 1000
        text = _clean(blk[m.end():])
        if text: out.append({'start': st, 'end': en, 'text': text})
    return out


def load_subs(path):
    txt = read_text(path)
    if path.lower().endswith('.smi') or '<sync' in txt.lower(): return parse_smi(txt)
    if path.lower().endswith(('.ass', '.ssa', '.vtt')):
        tmp = '/tmp/_conv.srt'
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', path, tmp], check=False)
        if os.path.exists(tmp): return parse_srt(read_text(tmp))
        return []
    return parse_srt(txt)


def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration:stream=index,codec_type,codec_name,width,height:stream_tags=language',
                        '-of', 'json', path], capture_output=True, text=True)
    j = json.loads(r.stdout or '{}')
    dur = float(j.get('format', {}).get('duration', 0) or 0)
    v = next((s for s in j.get('streams', []) if s.get('codec_type') == 'video'), {})
    subs = [s for s in j.get('streams', []) if s.get('codec_type') == 'subtitle']
    return {'dur': dur, 'vcodec': v.get('codec_name', ''), 'w': v.get('width', 0), 'h': v.get('height', 0), 'subs': subs}


def embedded_kor_subs(path, info):
    for s in info['subs']:
        lang = (s.get('tags') or {}).get('language', '')
        if s.get('codec_name') in ('subrip', 'ass', 'ssa', 'mov_text', 'webvtt', 'text') and re.match(r'(?i)ko', lang or ''):
            tmp = '/tmp/_emb.srt'
            subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', path, '-map', f"0:{s['index']}", '-f', 'srt', tmp], check=False)
            if os.path.exists(tmp): return parse_srt(read_text(tmp))
    return []


def to_srt(subs):
    def ts(t):
        t = max(0, t); ms = int(round(t * 1000))
        return f'{ms//3600000:02d}:{ms%3600000//60000:02d}:{ms%60000//1000:02d},{ms%1000:03d}'
    return '\n'.join(f'{i+1}\n{ts(s["start"])} --> {ts(s["end"])}\n{s["text"]}\n' for i, s in enumerate(subs))


def cut_720p(src, start, end, out, log=print):
    """원하는 구간을 720p H.264 mp4로 잘라요 (PC방 크롬에서 가볍게 열리는 형식)."""
    dur = end - start
    cmd = ['ffmpeg', '-v', 'error', '-stats_period', '5', '-progress', 'pipe:1', '-y',
           '-ss', str(start), '-i', src, '-t', str(dur),
           '-map', '0:v:0', '-map', '0:a:0?', '-sn', '-dn',
           '-vf', "scale=-2:'min(720,ih)'", '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '22', '-pix_fmt', 'yuv420p',
           '-c:a', 'aac', '-b:a', '160k', '-ac', '2', '-movflags', '+faststart', out]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    last = 0
    for line in p.stdout:
        if line.startswith('out_time_ms='):
            try:
                t = int(line.split('=')[1]) / 1e6
                if time.time() - last > 5:
                    log(f'  자르는 중… {min(100, t/dur*100):.0f}%'); last = time.time()
            except ValueError:
                pass
    p.wait()
    if p.returncode != 0 or not os.path.exists(out):
        raise RuntimeError('자르기에 실패했어요: ' + p.stderr.read()[-400:])


def make_material(movie, sub_path, start, end, out_root, log=print):
    info = probe(movie)
    if not info['dur']: raise RuntimeError('영화 파일을 읽지 못했어요.')
    if start is None or end is None or end <= start: raise RuntimeError('시작·끝 시간을 확인해 주세요. 예) 1:36:30 ~ 1:41:30')
    end = min(end, info['dur'])
    log(f'영화: {os.path.basename(movie)} · 전체 {hms(info["dur"])} · {info["vcodec"].upper()} {info["w"]}x{info["h"]}')
    subs = load_subs(sub_path) if sub_path else embedded_kor_subs(movie, info)
    log(f'자막: {len(subs)}줄' + ('' if subs else ' (자막이 없어요. 웹 도구에서 받아쓰기를 켜면 돼요)'))
    os.makedirs(out_root, exist_ok=True)
    base = re.sub(r'[\\/:*?"<>|]', '', os.path.splitext(os.path.basename(movie))[0])[:40]
    name = f'{base}_{hms(start).replace(":", "")}-{hms(end).replace(":", "")}'
    out_mp4 = os.path.join(out_root, name + '.mp4'); out_srt = os.path.join(out_root, name + '.srt')
    t0 = time.time(); log(f'{hms(start)} ~ {hms(end)} ({(end-start)/60:.1f}분) 720p로 자르는 중…')
    tmp = '/tmp/' + name + '.mp4'
    cut_720p(movie, start, end, tmp, log)
    clip = [{'start': s['start'] - start, 'end': s['end'] - start, 'text': s['text']} for s in subs if s['end'] > start and s['start'] < end]
    for s in clip: s['start'] = max(0, s['start'])
    open('/tmp/' + name + '.srt', 'w', encoding='utf-8').write(to_srt(clip))
    import shutil
    shutil.copy('/tmp/' + name + '.mp4', out_mp4); shutil.copy('/tmp/' + name + '.srt', out_srt)
    mb = os.path.getsize(out_mp4) / 1048576
    log(f'완료! ({time.time()-t0:.0f}초) 영상 {mb:.0f}MB · 자막 {len(clip)}줄')
    log(f'구글 드라이브 > {OUT_DIR_NAME} 폴더에 저장했어요:\n  {name}.mp4\n  {name}.srt')
    return out_mp4, out_srt

ROOT='/content/drive/MyDrive'
OUT=ROOT+'/'+OUT_DIR_NAME
print('드라이브에서 영화와 자막을 찾는 중…')
VIDS, SUBS = find_files(ROOT)
print('\n🎬 영화 파일')
for i,p in enumerate(VIDS,1): print(f'  {i}. {p[len(ROOT)+1:]}  ({os.path.getsize(p)/1073741824:.1f}GB)')
print('\n📝 자막 파일  (0 = 자막 파일 없음 · 영화 안에 든 한국어 자막을 써요)')
for i,p in enumerate(SUBS,1): print(f'  {i}. {p[len(ROOT)+1:]}')
if not VIDS: print('\n영화 파일을 못 찾았어요. 드라이브에 영화를 올렸는지 확인해 주세요.')


In [ ]:
#@title ② 구간 잘라서 재료 만들기 { display-mode: "form" }
영화_번호 = 1 #@param {type:"integer"}
자막_번호 = 1 #@param {type:"integer"}
시작 = "1:36:30" #@param {type:"string"}
끝 = "1:41:30" #@param {type:"string"}
try:
    movie = VIDS[영화_번호-1]
    sub = SUBS[자막_번호-1] if 자막_번호>0 else None
    make_material(movie, sub, parse_time(시작), parse_time(끝), OUT)
except NameError:
    print('먼저 위의 ① 칸 ▶ 버튼을 눌러 주세요.')
except IndexError:
    print('번호를 확인해 주세요. ① 칸 목록에 있는 번호를 넣어야 해요.')
except Exception as e:
    print('문제가 생겼어요:', e)
